# 16 - Mobile Robot Models

## What / Why / How

**What we are trying to do:** Model common mobile robot motion using unicycle and bicycle dynamics plus pure pursuit tracking.

**Why this matters:** Wheeled robots are everywhere, and their motion constraints determine which paths and controllers are realistic.

**How we will do it:** Implement simple motion models, follow a sinusoidal reference path, and compare with a bicycle-style trajectory.

## Goal

Understand common mobile robot models:

- Unicycle
- Differential drive
- Bicycle model
- Pure pursuit tracking

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
from robotics_mastery.control import pure_pursuit_control
from robotics_mastery.geometry import wrap_angle

def unicycle_step(pose, v, omega, dt=0.05):
    x, y, theta = pose
    return np.array([
        x + v*np.cos(theta)*dt,
        y + v*np.sin(theta)*dt,
        wrap_angle(theta + omega*dt),
    ])

def bicycle_step(state, speed, steering, wheelbase=0.35, dt=0.05):
    x, y, theta = state
    beta = np.tan(steering) / wheelbase
    return np.array([
        x + speed*np.cos(theta)*dt,
        y + speed*np.sin(theta)*dt,
        wrap_angle(theta + speed*beta*dt),
    ])

In [ ]:
path = np.array([[i * 0.25, np.sin(i * 0.25)] for i in range(45)])
pose = np.array([0.0, -0.5, 0.0])
traj = []
waypoint_idx = 0
for _ in range(300):
    while waypoint_idx < len(path)-1 and np.linalg.norm(path[waypoint_idx] - pose[:2]) < 0.5:
        waypoint_idx += 1
    v, omega = pure_pursuit_control(pose, path[waypoint_idx], lookahead=0.8, speed=0.5)
    pose = unicycle_step(pose, v, omega)
    traj.append(pose.copy())
traj = np.array(traj)
print("final pose:", traj[-1])
print("last waypoint:", path[-1])

In [ ]:
car = np.array([0.0, -0.5, 0.0])
car_traj = []
for _ in range(160):
    steering = 0.35 * np.sin(_ * 0.04)
    car = bicycle_step(car, speed=0.6, steering=steering)
    car_traj.append(car.copy())
car_traj = np.array(car_traj)
print("bicycle final:", car_traj[-1])

In [ ]:
if HAS_PLOT:
    plt.figure(figsize=(6, 4))
    plt.plot(path[:, 0], path[:, 1], "k--", label="reference")
    plt.plot(traj[:, 0], traj[:, 1], label="pure pursuit")
    plt.plot(car_traj[:, 0], car_traj[:, 1], label="bicycle sample")
    plt.axis("equal")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()
else:
    plot_unavailable()

## Exercises

1. Add wheel speed limits.
2. Increase lookahead and explain the tracking change.
3. Implement Stanley control and compare with pure pursuit.